# O Nordeste espera mais?

**Análise de prazo de entrega no e-commerce brasileiro — 100 mil pedidos reais**

Kevin Moreira da Costa · [github.com/KevinCosta-AI](https://github.com/KevinCosta-AI)

---

### A pergunta

Eu moro em Fortaleza. Quando compro pela internet, tenho a impressão de que espero mais que quem mora em São Paulo.

Impressão não é dado. Então vim medir.

### Os dados

Dataset público da **Olist**, marketplace brasileiro: cerca de **100 mil pedidos reais** feitos entre 2016 e 2018, com a data prometida, a data em que o pedido realmente chegou, o estado do cliente e a nota que ele deu depois.

### O plano

| Etapa | O que faz |
|---|---|
| **1. Esta parte** | Quanto cada região espera, e quem recebe atrasado |
| 2. Em breve | Um modelo que prevê se um pedido vai atrasar |
| 3. Em breve | Ler os comentários dos clientes e descobrir do que reclamam |

---
## Passo 1 — Conferir se os dados estão no ar

Antes de qualquer análise, a primeira coisa é checar se a fonte responde. Se um arquivo sumir do servidor, quero descobrir agora e com uma mensagem clara — não daqui a dez células, com um erro vermelho enorme.

In [ ]:
import pandas as pd

BASE = "https://raw.githubusercontent.com/spdrio/Brazilian-E-Commerce-Public-Dataset-by-Olist/master/files/"

ARQUIVOS = [
    "olist_orders_dataset.csv",
    "olist_customers_dataset.csv",
    "olist_order_reviews_dataset.csv",
    "olist_order_items_dataset.csv",
]

print("Testando a fonte de dados...\n")
ok = []
for nome in ARQUIVOS:
    try:
        # nrows=5 baixa só as primeiras linhas: é um teste rápido, não o download inteiro
        amostra = pd.read_csv(BASE + nome, nrows=5)
        print(f"  OK       {nome:42s} {len(amostra.columns)} colunas")
        ok.append(nome)
    except Exception as e:
        print(f"  FALHOU   {nome:42s} -> {type(e).__name__}")

print(f"\n{len(ok)} de {len(ARQUIVOS)} arquivos responderam.")
if len(ok) < len(ARQUIVOS):
    print("Manda print dessa saída pro Claude que a gente conserta a fonte.")

---
## Passo 2 — Carregar os pedidos e os clientes

Duas tabelas:

- **orders** — um pedido por linha: quando foi comprado, quando prometeram entregar, quando chegou de verdade
- **customers** — quem é o cliente e, o que me interessa aqui, **em que estado ele mora**

Elas se ligam pela coluna `customer_id`. Juntar as duas é o que me permite perguntar "quanto tempo esperou *cada estado*".

In [ ]:
# parse_dates converte texto em data de verdade, senão não dá pra subtrair uma da outra
pedidos = pd.read_csv(
    BASE + "olist_orders_dataset.csv",
    parse_dates=["order_purchase_timestamp",
                 "order_delivered_customer_date",
                 "order_estimated_delivery_date"],
)

clientes = pd.read_csv(BASE + "olist_customers_dataset.csv")

print(f"pedidos:  {len(pedidos):,} linhas".replace(",", "."))
print(f"clientes: {len(clientes):,} linhas".replace(",", "."))

pedidos.head(3)

---
## Passo 3 — Limpar

Nem todo pedido serve pra esta análise. Eu só posso medir tempo de entrega de pedido que **foi entregue** — pedido cancelado ou ainda a caminho não tem data de chegada, e incluir isso sujaria a média.

Essa é a parte que ninguém mostra no currículo e que é metade do trabalho real.

In [ ]:
antes = len(pedidos)

# só pedidos com status "entregue"
df = pedidos[pedidos["order_status"] == "delivered"].copy()

# e que realmente têm data de chegada preenchida
df = df.dropna(subset=["order_delivered_customer_date", "order_estimated_delivery_date"])

# junta o estado do cliente
df = df.merge(clientes[["customer_id", "customer_state"]], on="customer_id", how="left")

print(f"Comecei com {antes:,} pedidos".replace(",", "."))
print(f"Sobraram    {len(df):,} entregues e com data completa".replace(",", "."))
print(f"Descartei   {antes - len(df):,} ({(antes-len(df))/antes:.1%})".replace(",", "."))

---
## Passo 4 — Criar as duas medidas que importam

**`dias_entrega`** — da compra até a chegada. É o que o cliente sente como espera.

**`dias_atraso`** — da data prometida até a chegada. Positivo = chegou depois do combinado. Negativo = chegou antes.

Repara que são coisas diferentes: uma entrega pode demorar 20 dias e **não** estar atrasada, se a loja prometeu 25.

In [ ]:
df["dias_entrega"] = (df["order_delivered_customer_date"] - df["order_purchase_timestamp"]).dt.days
df["dias_atraso"]  = (df["order_delivered_customer_date"] - df["order_estimated_delivery_date"]).dt.days
df["atrasou"]      = df["dias_atraso"] > 0

REGIOES = {
    "Norte":        ["AC","AP","AM","PA","RO","RR","TO"],
    "Nordeste":     ["AL","BA","CE","MA","PB","PE","PI","RN","SE"],
    "Centro-Oeste": ["DF","GO","MT","MS"],
    "Sudeste":      ["ES","MG","RJ","SP"],
    "Sul":          ["PR","RS","SC"],
}
uf_para_regiao = {uf: reg for reg, ufs in REGIOES.items() for uf in ufs}
df["regiao"] = df["customer_state"].map(uf_para_regiao)

print(f"Espera média no Brasil:  {df['dias_entrega'].mean():.1f} dias")
print(f"Pedidos que atrasaram:   {df['atrasou'].mean():.1%}")

---
## Passo 5 — A resposta

Agora sim: quanto cada região espera?

In [ ]:
resumo = (
    df.groupby("regiao")
      .agg(pedidos=("order_id", "count"),
           espera_media=("dias_entrega", "mean"),
           taxa_atraso=("atrasou", "mean"))
      .sort_values("espera_media", ascending=False)
)

resumo_exibir = resumo.copy()
resumo_exibir["espera_media"] = resumo_exibir["espera_media"].round(1)
resumo_exibir["taxa_atraso"]  = (resumo_exibir["taxa_atraso"] * 100).round(1)
resumo_exibir.columns = ["pedidos", "espera média (dias)", "atrasaram (%)"]
resumo_exibir

### E o Ceará?

Média por estado. Destaquei o meu.

In [ ]:
import matplotlib.pyplot as plt

# ---- paleta ----
TINTA      = "#0b0b0b"   # texto principal
TINTA_FRACA= "#52514e"   # texto secundário
BARRA      = "#c9c8c1"   # barras normais
DESTAQUE   = "#2a78d6"   # a barra do Ceará
FUNDO      = "#fcfcfb"

por_uf = (df.groupby("customer_state")["dias_entrega"]
            .mean()
            .sort_values(ascending=True))

cores = [DESTAQUE if uf == "CE" else BARRA for uf in por_uf.index]

fig, ax = plt.subplots(figsize=(8, 9), facecolor=FUNDO)
ax.set_facecolor(FUNDO)

ax.barh(por_uf.index, por_uf.values, color=cores, height=0.72)

# valor direto na ponta da barra — dispensa eixo x
for uf, valor in por_uf.items():
    ax.text(valor + 0.35, uf, f"{valor:.0f}",
            va="center", fontsize=9,
            color=TINTA if uf == "CE" else TINTA_FRACA,
            fontweight="bold" if uf == "CE" else "normal")

ax.set_title("Dias até a entrega, por estado do cliente",
             fontsize=13, color=TINTA, pad=16, loc="left", fontweight="bold")
subtitulo = f"Média de dias entre a compra e a chegada · {len(df):,} pedidos entregues · Olist 2016–2018".replace(",", ".")
ax.text(0, 1.015, subtitulo, transform=ax.transAxes, fontsize=9, color=TINTA_FRACA)

# eixos recessivos: o gráfico é a informação, a moldura não
ax.set_xlabel("")
ax.set_xticks([])
for lado in ["top", "right", "bottom", "left"]:
    ax.spines[lado].set_visible(False)
ax.tick_params(axis="y", length=0, labelsize=9, colors=TINTA_FRACA)
ax.margins(x=0.08)

plt.tight_layout()
plt.show()

---
## Passo 6 — A pergunta que muda tudo: quanto eles PROMETERAM?

Até aqui eu só sei que o Nordeste espera mais. Isso sozinho não prova nada — o Nordeste **é** mais longe dos centros de distribuição, então demorar mais é esperado.

A pergunta certa é outra: **a promessa já leva a distância em conta?**

Se prometem 30 dias pra Fortaleza e 21 pra São Paulo, a distância já está embutida — e aí ela não pode ser a explicação para a promessa quebrar mais.

In [ ]:
df["dias_prometidos"] = (df["order_estimated_delivery_date"] - df["order_purchase_timestamp"]).dt.days

promessa = (
    df.groupby("regiao")
      .agg(prometido=("dias_prometidos", "mean"),
           real=("dias_entrega", "mean"),
           atrasaram=("atrasou", "mean"))
      .sort_values("prometido", ascending=False)
)
promessa["folga"] = promessa["prometido"] - promessa["real"]
promessa["atrasaram"] = promessa["atrasaram"] * 100
promessa.round(1)

**A distância já está na promessa.** Prometem 30,3 dias para o Nordeste e 21,2 para o Sudeste — nove dias a mais, exatamente porque é mais longe.

Então "é longe" não explica por que a promessa quebra o dobro das vezes no Nordeste. Precisa haver outra coisa.

In [ ]:
import matplotlib.pyplot as plt

TINTA="#0b0b0b"; TINTA_FRACA="#52514e"; CINZA="#c9c8c1"
AZUL="#2a78d6"; LARANJA="#eb6834"; FUNDO="#fcfcfb"

def haltere(dados, col_a, col_b, rotulo_a, rotulo_b, titulo, subtitulo, unidade="dias"):
    """Gráfico de haltere: duas bolinhas ligadas por uma linha, uma linha por categoria.
    Serve pra comparar duas medidas na MESMA unidade sem precisar de dois eixos."""
    d = dados.sort_values(col_b)
    fig, ax = plt.subplots(figsize=(9, 4.0), facecolor=FUNDO)
    ax.set_facecolor(FUNDO)

    for y, (nome, linha) in enumerate(d.iterrows()):
        ax.plot([linha[col_a], linha[col_b]], [y, y], color=CINZA, lw=2, zorder=1)
        ax.scatter(linha[col_a], y, s=110, color=AZUL,    zorder=2, edgecolor=FUNDO, linewidth=2)
        ax.scatter(linha[col_b], y, s=110, color=LARANJA, zorder=2, edgecolor=FUNDO, linewidth=2)
        ax.text(linha[col_a] - 1.0, y, f"{linha[col_a]:.0f}", va="center", ha="right", fontsize=9, color=TINTA_FRACA)
        ax.text(linha[col_b] + 1.0, y, f"{linha[col_b]:.0f}", va="center", ha="left",  fontsize=9, color=TINTA_FRACA)

    ax.set_yticks(range(len(d)))
    ax.set_yticklabels(d.index, fontsize=10, color=TINTA)
    ax.set_title(titulo, fontsize=13, color=TINTA, pad=40, loc="left", fontweight="bold")
    ax.text(0, 1.06, subtitulo, transform=ax.transAxes, fontsize=9, color=TINTA_FRACA)

    ax.scatter([], [], s=110, color=AZUL,    label=rotulo_a)
    ax.scatter([], [], s=110, color=LARANJA, label=rotulo_b)
    leg = ax.legend(loc="lower right", frameon=False, fontsize=9, ncol=2)
    for t in leg.get_texts(): t.set_color(TINTA_FRACA)

    ax.set_xticks([])
    for lado in ["top","right","bottom","left"]: ax.spines[lado].set_visible(False)
    ax.tick_params(axis="y", length=0)
    ax.margins(x=0.12, y=0.22)
    plt.tight_layout(); plt.show()

haltere(promessa, "real", "prometido", "Chegou em", "Prometeram",
        "A distância já está no prazo prometido",
        "Média de dias · por região do cliente · 96.470 pedidos entregues")

---
## Passo 7 — Então por quê? A resposta está na variação

Média esconde coisa. Duas regiões podem ter a mesma média e experiências completamente diferentes — uma entregando sempre no mesmo prazo, outra ora rápido ora péssimo.

Então vou olhar não a média, mas **o azar**: em quantos dias chega a metade dos pedidos (mediana), e em quantos dias chega o pedido do décimo mais lento (percentil 90).

A distância entre esses dois números é o tamanho do risco.

In [ ]:
variacao = (
    df.groupby("regiao")["dias_entrega"]
      .agg(mediana="median",
           p90=lambda s: s.quantile(0.90),
           desvio="std")
      .sort_values("desvio", ascending=False)
)
variacao["azar"] = variacao["p90"] - variacao["mediana"]
variacao.round(1)

In [ ]:
haltere(variacao, "mediana", "p90", "Metade chega em", "1 em cada 10 leva",
        "No Nordeste, o azar custa muito mais caro",
        "Mediana e percentil 90 dos dias até a entrega · por região")

**Aqui está a explicação.**

No **Sudeste** a entrega é previsível: metade chega em 8 dias e, mesmo no azar, 19. Dá pra prometer com segurança.

No **Nordeste** não: metade chega em 17 dias, mas quando dá errado vira 33 — o dobro.

Prazo imprevisível quebra promessa. Não importa quanta folga se dê na média: se a variação for grande, uma fatia dos pedidos estoura qualquer prazo razoável. É por isso que 12,7% atrasam mesmo com 30 dias prometidos.

---
## O que eu achei

**A espera média no Brasil foi de 12,1 dias, e 6,8% dos pedidos chegaram depois da data prometida.**

- O estado que mais espera é **Roraima, com 29 dias**. O que menos espera é **São Paulo, com 8**.
- O **Ceará espera 21 dias** — duas vezes e meia São Paulo.
- Por região, o **Nordeste espera 19,5 dias** contra 10,3 do Sudeste, e **12,7% dos pedidos chegam atrasados** — mais que o dobro dos 6,1% do Sudeste.

**E a explicação não é a distância.**

Foi o que eu achei que fosse, no começo. Mas quando comparei o prazo *prometido* com o prazo *real*, vi que prometem 30,3 dias para o Nordeste e 21,2 para o Sudeste. A distância já está embutida na promessa — logo, ela não pode explicar por que a promessa quebra o dobro das vezes.

O que explica é a **variação**. No Sudeste, metade dos pedidos chega em 8 dias e o décimo mais lento em 19. No Nordeste, metade chega em 17 e o décimo mais lento em 33 — o dobro da mediana. A entrega para o Nordeste não é só mais lenta: é **menos previsível**.

E prazo imprevisível quebra promessa. Por mais folga que se dê na média, se a cauda for longa, uma fatia dos pedidos estoura qualquer prazo razoável.

**Por que isso importa para quem opera entrega:** melhorar a média no Nordeste ajuda pouco. O que reduz atraso ali é encurtar a cauda — atacar os pedidos que dão errado, não os que já vão bem. E enquanto a cauda for longa, o prazo prometido devia refletir isso, em vez de prometer o improvável.

---

## Como cheguei aqui

| Passo | O que fiz |
|---|---|
| 1 | Conferi que as quatro fontes de dados respondiam |
| 2 | Carreguei 99.441 pedidos e 99.441 clientes, e juntei as duas tabelas |
| 3 | Fiquei só com pedidos entregues e com data completa: 96.470 (descartei 3%) |
| 4 | Criei as medidas: dias até a entrega, dias de atraso, região do cliente |
| 5 | Comparei espera e taxa de atraso por região e por estado |
| 6 | Comparei o prazo prometido com o real — o que derrubou a hipótese da distância |
| 7 | Olhei a variação (mediana vs percentil 90) — que deu a resposta |

---

## Próximos passos

**Parte 2 — prever.** Com o que dá pra saber no momento da compra (estado, valor, tipo de produto, vendedor), consigo prever quais pedidos vão atrasar? É um problema de classificação, e é onde entra o machine learning.

**Parte 3 — entender.** A Olist guardou os comentários escritos pelos clientes, em português. Quando a entrega atrasa, do que exatamente a pessoa reclama? Ler 100 mil comentários na mão é impossível — é aí que entra IA.

---

*Dados: [Brazilian E-Commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) · 2016–2018 · uso público.*